# Gender Detection Pipeline

Copy/run each cell below in order in Google Colab.

**Before you start:** Runtime > Change runtime type > Hardware accelerator > T4 GPU > Save

Every numbered step below has: a markdown header, a code cell, and (where useful) an **Example output** cell right after it showing what you should expect to see.

### CELL 1: Confirm GPU is on

In [ ]:
import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
print("GPU available:", gpus)
if not gpus:
    print("WARNING: No GPU detected. Go to Runtime > Change runtime type > GPU, then restart.")

### CELL 2: Install dependencies

In [ ]:
!pip install inaSpeechSegmenter -q

### CELL 3: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### CELL 4: Configuration — EDIT THESE PATHS

In [ ]:
import os

# Folder in your Drive where all the mp3 audio files are (or subfolders of them)
AUDIO_DIR = '/content/drive/MyDrive/colab_notebooks/masculine_default/audio'

# Local Colab disk (fast, temporary) for the trimmed 30-second clips
TRIMMED_DIR = '/content/trimmed_audio'

# Where results and errors get saved (on Drive, so they survive disconnects)
RESULTS_CSV = '/content/drive/MyDrive/colab_notebooks/masculine_default/gender_results.csv'
ERROR_LOG = '/content/drive/MyDrive/colab_notebooks/masculine_default/gender_errors.csv'

# NEW: raw segment-level output (every label inaSpeechSegmenter returns)
SEGMENTS_CSV = '/content/drive/MyDrive/colab_notebooks/masculine_default/gender_segments.csv'

# NEW: 30s vs full-audio validation output
VALIDATION_CSV = '/content/drive/MyDrive/colab_notebooks/masculine_default/gender_validation.csv'

# NEW: optional manual annotation file (you create this yourself if you want manual validation)
MANUAL_ANNOTATIONS_CSV = '/content/drive/MyDrive/colab_notebooks/masculine_default/manual_gender_annotations.csv'

# NEW: how many files to use for the 30s-vs-full-audio validation (NOT the whole corpus)
VALIDATION_SAMPLE_SIZE = 50
VALIDATION_SEED = 42  # keep this fixed so the sample is reproducible

CLIP_SECONDS = 30  # matches the paper's methodology (r=0.79-0.82 correlation with full episode)

os.makedirs(TRIMMED_DIR, exist_ok=True)

### CELL 5: List files & resume support

In [ ]:
import glob
import pandas as pd

all_files = glob.glob(os.path.join(AUDIO_DIR, '**', '*.mp3'), recursive=True)
print(f"Found {len(all_files)} audio files")

if os.path.exists(RESULTS_CSV):
    done_df = pd.read_csv(RESULTS_CSV)
    done_ids = set(done_df['file'])
    print(f"Resuming: {len(done_ids)} files already processed, skipping them")
else:
    done_ids = set()

files_to_process = [f for f in all_files if os.path.basename(f) not in done_ids]
print(f"{len(files_to_process)} files left to process")

### CELL 6: Trim to first 30s (parallel, fast)

In [ ]:
import subprocess
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm

def trim_audio(src_path):
    filename = os.path.basename(src_path)
    dst_path = os.path.join(TRIMMED_DIR, filename)
    if os.path.exists(dst_path):
        return filename, True, None
    try:
        # -c copy = stream copy, NO re-encoding = very fast (just cuts the container)
        cmd = ['ffmpeg', '-y', '-i', src_path, '-t', str(CLIP_SECONDS), '-c', 'copy', dst_path]
        result = subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.PIPE, timeout=30)
        if result.returncode != 0:
            # fallback: some files can't be stream-copied cleanly, so re-encode instead
            cmd2 = ['ffmpeg', '-y', '-i', src_path, '-t', str(CLIP_SECONDS), dst_path]
            result2 = subprocess.run(cmd2, stdout=subprocess.DEVNULL, stderr=subprocess.PIPE, timeout=60)
            if result2.returncode != 0:
                return filename, False, result2.stderr.decode(errors='ignore')[:300]
        return filename, True, None
    except Exception as e:
        return filename, False, str(e)

trim_errors = []
with ThreadPoolExecutor(max_workers=8) as executor:
    futures = {executor.submit(trim_audio, f): f for f in files_to_process}
    for future in tqdm(as_completed(futures), total=len(futures), desc="Trimming audio"):
        filename, success, err = future.result()
        if not success:
            trim_errors.append({'file': filename, 'stage': 'trim', 'error': err})

print(f"Trimming done. {len(trim_errors)} failures.")
if trim_errors:
    pd.DataFrame(trim_errors).to_csv(ERROR_LOG, mode='a', index=False,
                                      header=not os.path.exists(ERROR_LOG))

### CELL 7: Load the gender detection model

In [ ]:
from inaSpeechSegmenter import Segmenter

seg = Segmenter(vad_engine='smn', detect_gender=True)
print("Model loaded. Ready to run on GPU.")

### CELL 8: Run gender detection with checkpointing (+ saves ALL segment labels)

**`gender_results.csv`** (main pipeline output — one row per file):

| file | male_seconds | female_seconds | dominant_gender |
|---|---|---|---|
| RJRaunakk__abc123.mp3 | 25.9 | 0.0 | male |
| SpiritualTalksHi__def456.mp3 | 3.2 | 21.4 | female |
| GamingZoneHindi__ghi789.mp3 | 0.0 | 0.0 | unknown |

**`gender_segments.csv`** (raw segment-level output — every label, every file):

| file | segment_id | label | start | end | duration |
|---|---|---|---|---|---|
| RJRaunakk__abc123.mp3 | 1 | male | 0.00 | 4.20 | 4.20 |
| RJRaunakk__abc123.mp3 | 2 | music | 4.20 | 6.80 | 2.60 |
| RJRaunakk__abc123.mp3 | 3 | male | 6.80 | 28.50 | 21.70 |
| SpiritualTalksHi__def456.mp3 | 1 | female | 0.00 | 21.40 | 21.40 |
| SpiritualTalksHi__def456.mp3 | 2 | noEnergy | 21.40 | 30.00 | 8.60 |
| GamingZoneHindi__ghi789.mp3 | 1 | noEnergy | 0.00 | 30.00 | 30.00 |

In [ ]:
import time

CHECKPOINT_EVERY = 50

# If True, prints the full segment-by-segment breakdown for every file (like your example).
# If the Colab tab starts feeling slow/laggy after a few thousand files, set this to False -
# every detail still gets saved to VERBOSE_LOG_FILE either way, you'd just stop watching it live.
PRINT_TO_SCREEN = True
VERBOSE_LOG_FILE = '/content/drive/MyDrive/colab_notebooks/masculine_default/gender_detection_log.txt'

results_buffer = []
error_buffer = []
segments_buffer = []  # NEW: raw segment-level rows (all labels, not just male/female)

trimmed_files = [
    os.path.join(TRIMMED_DIR, os.path.basename(f)) for f in files_to_process
    if os.path.exists(os.path.join(TRIMMED_DIR, os.path.basename(f)))
]

def get_display_name(filename):
    # filenames look like "channelName__videoID.mp3" -> just show the channel name
    name = filename.rsplit('.', 1)[0]
    return name.split('__')[0] if '__' in name else name

def log_line(f_handle, text):
    if PRINT_TO_SCREEN:
        print(text)
    f_handle.write(text + '\n')

# NEW: flush segments_buffer to SEGMENTS_CSV, same checkpoint pattern as results/errors
def flush_segments_buffer():
    global segments_buffer
    if segments_buffer:
        pd.DataFrame(segments_buffer).to_csv(SEGMENTS_CSV, mode='a', index=False,
                                              header=not os.path.exists(SEGMENTS_CSV))
        segments_buffer = []

start_time = time.time()
log_f = open(VERBOSE_LOG_FILE, 'a', encoding='utf-8')

for i, path in enumerate(tqdm(trimmed_files, desc="Gender detection")):
    filename = os.path.basename(path)
    try:
        segments = seg(path)  # list of (label, start_time, end_time)

        # NEW: record every segment returned (male, female, music, noEnergy, noise, ...)
        for seg_id, (label, start, end) in enumerate(segments, start=1):
            segments_buffer.append({
                'file': filename,
                'segment_id': seg_id,
                'label': label,
                'start': round(start, 2),
                'end': round(end, 2),
                'duration': round(end - start, 2)
            })

        male_time = sum(end - start for label, start, end in segments if label == 'male')
        female_time = sum(end - start for label, start, end in segments if label == 'female')
        total_speech = male_time + female_time

        if total_speech == 0:
            dominant = 'unknown'
        else:
            dominant = 'male' if male_time >= female_time else 'female'
        male_pct = (male_time / total_speech * 100) if total_speech > 0 else 0.0
        female_pct = (female_time / total_speech * 100) if total_speech > 0 else 0.0

        # ---- print/log in the segment-by-segment format you asked for ----
        log_line(log_f, f"\nProcessing: {get_display_name(filename)}")
        log_line(log_f, "  Segments:")
        for label, start, end in segments:
            log_line(log_f, f"    {label:<12} {start:.1f}s -> {end:.1f}s ({end - start:.1f}s)")
        log_line(log_f, "  Summary:")
        log_line(log_f, f"    Male   : {male_time:.1f}s ({male_pct:.1f}%)")
        log_line(log_f, f"    Female : {female_time:.1f}s ({female_pct:.1f}%)")
        log_line(log_f, f"    Result : {dominant.upper()}")

        results_buffer.append({
            'file': filename,
            'male_seconds': round(male_time, 2),
            'female_seconds': round(female_time, 2),
            'dominant_gender': dominant
        })
    except Exception as e:
        error_buffer.append({'file': filename, 'stage': 'gender_detection', 'error': str(e)[:300]})
        log_line(log_f, f"\nProcessing: {get_display_name(filename)}  -->  ERROR: {str(e)[:150]}")

    # save progress periodically so a crash/disconnect doesn't lose everything
    if (i + 1) % CHECKPOINT_EVERY == 0:
        log_f.flush()
        if results_buffer:
            pd.DataFrame(results_buffer).to_csv(RESULTS_CSV, mode='a', index=False,
                                                 header=not os.path.exists(RESULTS_CSV))
            results_buffer = []
        if error_buffer:
            pd.DataFrame(error_buffer).to_csv(ERROR_LOG, mode='a', index=False,
                                               header=not os.path.exists(ERROR_LOG))
            error_buffer = []
        flush_segments_buffer()  # NEW

# flush anything left in the buffer at the end
if results_buffer:
    pd.DataFrame(results_buffer).to_csv(RESULTS_CSV, mode='a', index=False,
                                         header=not os.path.exists(RESULTS_CSV))
if error_buffer:
    pd.DataFrame(error_buffer).to_csv(ERROR_LOG, mode='a', index=False,
                                       header=not os.path.exists(ERROR_LOG))
flush_segments_buffer()  # NEW

elapsed = time.time() - start_time
log_f.close()
print(f"\nDone. Processed {len(trimmed_files)} files in {elapsed/60:.1f} minutes")
if trimmed_files:
    print(f"Average: {elapsed/len(trimmed_files):.2f} sec/file")

### CELL 9: Clean up local disk

In [ ]:
# Run this only after you've confirmed RESULTS_CSV on Drive looks correct.
# It just clears Colab's temporary disk, not your actual audio files.
import shutil
shutil.rmtree(TRIMMED_DIR, ignore_errors=True)
# print("Cleaned up temporary trimmed audio.")

### CELL 10: (diagnostic) Listen to one sample before retrying everything

In [ ]:
current_df = pd.read_csv(RESULTS_CSV)
unknown_files = current_df[current_df["dominant_gender"] == "unknown"]["file"].tolist()
print(f"{len(unknown_files)} files currently marked unknown")

unknown_check = current_df[current_df["dominant_gender"] == "unknown"].copy()
unknown_check["channel"] = unknown_check["file"].apply(lambda f: f.split("__")[0] if "__" in f else f)
print("\nTop affected channels:")
print(unknown_check["channel"].value_counts().head(10))

CHECK_CHANNEL = "ShibuThomas"  # change to whichever channel you want to spot-check
sample_files = [f for f in unknown_files if CHECK_CHANNEL in f]
if sample_files:
    sample_path = glob.glob(os.path.join(AUDIO_DIR, "**", sample_files[0]), recursive=True)
    if sample_path:
        import shutil
        shutil.copy(sample_path[0], "/content/sample_check.mp3")
        print(f"\nCopied {sample_files[0]} to /content/sample_check.mp3 - download and listen")
    else:
        print("File not found on disk - check AUDIO_DIR path")
else:
    print(f"No unknown files found for channel {CHECK_CHANNEL}")

#Cell 10A


In [ ]:
# ============================================================
# CELL 10A: Retry unknown files with 30-90 second window
# Place this AFTER Cell 10 (unknown count check)
# ============================================================

import subprocess
import os
import pandas as pd
from tqdm import tqdm
from inaSpeechSegmenter import Segmenter

# load current results
results_df = pd.read_csv(RESULTS_CSV)
unknown_df = results_df[results_df["dominant_gender"] == "unknown"].copy()

print(f"Total videos        : {len(results_df)}")
print(f"Unknown before retry: {len(unknown_df)}")

if len(unknown_df) == 0:
    print("No unknowns to retry!")
else:
    seg = Segmenter()

    RETRY_START    = 30   # skip first 30 seconds (intro/music)
    RETRY_DURATION = 120   # take next 60 seconds (30→90)
    RETRY_CLIP_DIR = "/content/retry_clips"
    os.makedirs(RETRY_CLIP_DIR, exist_ok=True)

    fixed   = 0
    still_unknown = 0

    for idx, row in tqdm(unknown_df.iterrows(), total=len(unknown_df), desc="Retrying unknowns"):
        filename = row["file"]
        src_path = os.path.join(AUDIO_DIR, filename)

        # search in subfolders too
        if not os.path.exists(src_path):
            import glob
            matches = glob.glob(
                os.path.join(AUDIO_DIR, "**", filename),
                recursive=True
            )
            src_path = matches[0] if matches else None

        if not src_path or not os.path.exists(src_path):
            print(f"  Audio not found: {filename}")
            continue

        retry_clip = os.path.join(RETRY_CLIP_DIR, f"retry_{filename}")

        try:
            # extract seconds 30-120
            subprocess.run([
                "ffmpeg", "-y",
                "-i", src_path,
                "-ss", str(RETRY_START),
                "-t",  str(RETRY_DURATION),
                "-ar", "16000",
                "-ac", "1",
                "-loglevel", "error",
                retry_clip
            ], check=True, capture_output=True, timeout=60)

            # run inaSpeechSegmenter on the 30-120 second clip
            segments = seg(retry_clip)

            male_sec   = sum(e - s for l, s, e in segments if l == "male")
            female_sec = sum(e - s for l, s, e in segments if l == "female")
            total      = male_sec + female_sec

            # clean up retry clip
            if os.path.exists(retry_clip):
                os.remove(retry_clip)

            if total > 0:
                if male_sec >= female_sec:
                    new_gender = "male"
                else:
                    new_gender = "female"

                # update results dataframe
                results_df.loc[
                    results_df["file"] == filename,
                    ["male_seconds", "female_seconds", "dominant_gender"]
                ] = [round(male_sec, 2), round(female_sec, 2), new_gender]

                fixed += 1
                print(f"  ✅ Fixed: {filename} → {new_gender} "
                      f"(M:{male_sec:.1f}s F:{female_sec:.1f}s)")
            else:
                still_unknown += 1

        except Exception as e:
            print(f"  ❌ Error on {filename}: {e}")
            if os.path.exists(retry_clip):
                os.remove(retry_clip)

    # save updated results back to Drive
    results_df.to_csv(RESULTS_CSV, index=False)

    print(f"\n{'='*50}")
    print(f"RETRY COMPLETE")
    print(f"{'='*50}")
    print(f"Fixed by retry      : {fixed}")
    print(f"Still unknown       : {still_unknown}")
    print(f"Updated CSV saved   : {RESULTS_CSV}")

### CELL 12: Summary — how many are still unknown after the retry

In [ ]:
still_unknown_path = os.path.join(os.path.dirname(RESULTS_CSV), "still_unknown_after_retry.csv")

if os.path.exists(still_unknown_path):
    still_unknown_df = pd.read_csv(still_unknown_path)
    print(f"Total still unknown after retry: {len(still_unknown_df)}")

    still_unknown_df["channel"] = still_unknown_df["file"].apply(lambda f: f.split("__")[0] if "__" in f else f)
    print("\nBy channel:")
    print(still_unknown_df["channel"].value_counts().to_string())
else:
    print("No file found - either the retry hasn't been run yet, or nothing remained unknown after it")

# also show the final overall count directly from the results CSV, as a cross-check
final_df = pd.read_csv(RESULTS_CSV)
print(f"\nCross-check - current gender_results.csv breakdown:")
print(final_df["dominant_gender"].value_counts())
print(f"\nUnknown as % of total: {(final_df['dominant_gender']=='unknown').sum() / len(final_df) * 100:.1f}%")

### CELL 13 (NEW): 30-second vs full-audio validation

Runs inaSpeechSegmenter on BOTH the first 30 seconds AND the full ~10-minute audio, for a small random sample of files (`VALIDATION_SAMPLE_SIZE`). This is separate from the main pipeline above and does NOT touch `gender_results.csv`.

Only run this once you're happy with the main run — it re-processes full audio, which is slower.

**`gender_validation.csv`** (30s estimate vs full ~10-minute audio, for the validation sample):

| file | male_seconds_30 | female_seconds_30 | male_seconds_10min | female_seconds_10min | dominant_gender_30 | dominant_gender_10min |
|---|---|---|---|---|---|---|
| RJRaunakk__abc123.mp3 | 25.9 | 0.0 | 412.3 | 18.7 | male | male |
| SpiritualTalksHi__def456.mp3 | 3.2 | 21.4 | 45.1 | 289.6 | female | female |
| GamingZoneHindi__ghi789.mp3 | 12.0 | 10.5 | 201.4 | 188.2 | male | male |

Printed correlations look like:
```
Male Pearson correlation:    r=0.812, p=0.0000
Female Pearson correlation:  r=0.774, p=0.0000
Male Spearman correlation:   rho=0.795, p=0.0000
Female Spearman correlation: rho=0.762, p=0.0000
```

In [ ]:
# NEW: 30-second vs 10-minute validation
import random
from scipy.stats import pearsonr, spearmanr

random.seed(VALIDATION_SEED)

# resume support: skip files already validated
if os.path.exists(VALIDATION_CSV):
    validated_df = pd.read_csv(VALIDATION_CSV)
    validated_ids = set(validated_df['file'])
    print(f"Resuming validation: {len(validated_ids)} files already validated")
else:
    validated_ids = set()

# sample from the full corpus (all_files), not just files_to_process, so validation
# isn't biased toward files that happened to be unprocessed
candidate_files = [f for f in all_files if os.path.basename(f) not in validated_ids]
sample_files = random.sample(candidate_files, min(VALIDATION_SAMPLE_SIZE, len(candidate_files)))
print(f"Validating on {len(sample_files)} files (target sample size: {VALIDATION_SAMPLE_SIZE})")

validation_results = []
validation_errors = []

def get_gender_seconds(segments):
    male_time = sum(end - start for label, start, end in segments if label == 'male')
    female_time = sum(end - start for label, start, end in segments if label == 'female')
    total = male_time + female_time
    if total == 0:
        dominant = 'unknown'
    else:
        dominant = 'male' if male_time >= female_time else 'female'
    return male_time, female_time, dominant

for src_path in tqdm(sample_files, desc="Validation (30s + full audio)"):
    filename = os.path.basename(src_path)
    clip_30_path = f"/content/val_30_{filename}"
    try:
        # A. first 30 seconds (reuse already-trimmed clip if it exists, else trim fresh)
        existing_trim = os.path.join(TRIMMED_DIR, filename)
        if os.path.exists(existing_trim):
            clip_30_path = existing_trim
        else:
            cmd30 = ['ffmpeg', '-y', '-i', src_path, '-t', str(CLIP_SECONDS), clip_30_path]
            subprocess.run(cmd30, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, timeout=60)

        segments_30 = seg(clip_30_path)
        male_30, female_30, dom_30 = get_gender_seconds(segments_30)

        # B. full ~10-minute audio, run directly on the original file (no trimming)
        segments_full = seg(src_path)
        male_full, female_full, dom_full = get_gender_seconds(segments_full)

        validation_results.append({
            'file': filename,
            'male_seconds_30': round(male_30, 2),
            'female_seconds_30': round(female_30, 2),
            'male_seconds_10min': round(male_full, 2),
            'female_seconds_10min': round(female_full, 2),
            'dominant_gender_30': dom_30,
            'dominant_gender_10min': dom_full
        })

        # clean up a freshly-trimmed 30s clip (don't delete files from TRIMMED_DIR, those are reused elsewhere)
        if clip_30_path != existing_trim if 'existing_trim' in dir() else True:
            pass
        if clip_30_path.startswith('/content/val_30_') and os.path.exists(clip_30_path):
            os.remove(clip_30_path)

    except Exception as e:
        validation_errors.append({'file': filename, 'stage': 'validation', 'error': str(e)[:300]})
        print(f"Error validating {filename}: {e}")

# save validation results (append, so re-runs with a different sample just extend the file)
if validation_results:
    pd.DataFrame(validation_results).to_csv(VALIDATION_CSV, mode='a', index=False,
                                             header=not os.path.exists(VALIDATION_CSV))
if validation_errors:
    pd.DataFrame(validation_errors).to_csv(ERROR_LOG, mode='a', index=False,
                                            header=not os.path.exists(ERROR_LOG))

print(f"\nValidation done. {len(validation_results)} succeeded, {len(validation_errors)} failed.")

# NEW: correlation between 30s estimate and full ~10-minute audio
val_df = pd.read_csv(VALIDATION_CSV)
if len(val_df) >= 2:
    male_pearson_r, male_pearson_p = pearsonr(val_df['male_seconds_30'], val_df['male_seconds_10min'])
    female_pearson_r, female_pearson_p = pearsonr(val_df['female_seconds_30'], val_df['female_seconds_10min'])
    male_spearman_r, male_spearman_p = spearmanr(val_df['male_seconds_30'], val_df['male_seconds_10min'])
    female_spearman_r, female_spearman_p = spearmanr(val_df['female_seconds_30'], val_df['female_seconds_10min'])

    print(f"\nValidation sample size: {len(val_df)}")
    print(f"Male Pearson correlation:    r={male_pearson_r:.3f}, p={male_pearson_p:.4f}")
    print(f"Female Pearson correlation:  r={female_pearson_r:.3f}, p={female_pearson_p:.4f}")
    print(f"Male Spearman correlation:   rho={male_spearman_r:.3f}, p={male_spearman_p:.4f}")
    print(f"Female Spearman correlation: rho={female_spearman_r:.3f}, p={female_spearman_p:.4f}")
else:
    print("Not enough validated files yet to compute correlations (need at least 2).")